# Testing different hyperparamters for SAKT to achive the best AUC possible

## Clone the repo

In [4]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git
%cd knowledge-tracing-collection-pytorch

!pip install -r requirements.txt

!grep -n "learning_rate\|hidden_size\|num_epochs\|batch_size" train.py
!grep -n "hidden_size\|embed\|d_model" models/sakt.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
fatal: destination path 'knowledge-tracing-collection-pytorch' already exists and is not an empty directory.
/content/knowledge-tracing-collection-pytorch
40:    batch_size = train_config["batch_size"]
41:    num_epochs = train_config["num_epochs"]
43:    learning_rate = train_config["learning_rate"]
109:        train_dataset, batch_size=batch_size, shuffle=True,
113:        test_dataset, batch_size=test_size, shuffle=True,
118:        opt = SGD(model.parameters(), learning_rate, momentum=0.9)
120:        opt = Adam(model.parameters(), learning_rate)
124:            train_loader, test_loader, num_epochs, opt, ckpt_path


## Import dataset

In [3]:
import os

os.makedirs("datasets/ASSIST2009", exist_ok=True)

import shutil

source = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/skill_builder_data.csv"
destination = "/content/knowledge-tracing-collection-pytorch/datasets/ASSIST2009/skill_builder_data.csv"

shutil.copy2(source, destination)

print("Copied:", os.path.exists(destination))

import pandas as pd

path = "datasets/ASSIST2009/skill_builder_data.csv"

## Dataset only works in latin1, so first read in latin1 then convert to utf-8 to be used by SAKT
df = pd.read_csv(path, encoding="latin-1")

print(df.shape)
print(df.columns.tolist())

df.to_csv(
    "datasets/ASSIST2009/skill_builder_data_utf8.csv",
    index=False,
    encoding="utf-8"
)

!mv datasets/ASSIST2009/skill_builder_data.csv datasets/ASSIST2009/skill_builder_data_original.csv
!mv datasets/ASSIST2009/skill_builder_data_utf8.csv datasets/ASSIST2009/skill_builder_data.csv

pd.read_csv(
    "datasets/ASSIST2009/skill_builder_data.csv"
).head()

Copied: True


/tmp/ipykernel_36897/3498793680.py:19: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, encoding="latin-1")


(525534, 30)
['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'bottom_hint', 'opportunity', 'opportunity_original']


/tmp/ipykernel_36897/3498793680.py:33: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(


,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,33022537,277618,64525,33139,51424,1,1,1,32454,tutor,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,33022709,277618,64525,33150,51435,1,1,1,4922,tutor,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,35450204,220674,70363,33159,51444,1,0,2,25390,tutor,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,35450295,220674,70363,33110,51395,1,1,1,4859,tutor,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,35450311,220674,70363,33196,51481,1,0,14,19813,tutor,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0


## Set up 6 experiments with different hyperparameters




In [35]:
import os
import json
import pickle
import pandas as pd

REPO_DIR = "/content/knowledge-tracing-collection-pytorch"
CONFIG_PATH = os.path.join(REPO_DIR, "config.json")

%cd /content/knowledge-tracing-collection-pytorch

# Save original config
with open(CONFIG_PATH, "r") as f:
    original_config = json.load(f)

# Six experiments
experiments = [
    {"learning_rate": 0.001, "d": 65},
    {"learning_rate": 0.001, "d": 125},
    {"learning_rate": 0.0005, "d": 65},
    {"learning_rate": 0.0005, "d": 125},
    {"learning_rate": 0.0001, "d": 65},
    {"learning_rate": 0.0001, "d": 125},
]

# Results saved to Drive
RESULTS_PATH = (
    "/content/drive/MyDrive/"
    "education-ml-research/"
    "sakt_hyperparameter_results.csv"
)

results = []

print("Setup complete.")

/content/knowledge-tracing-collection-pytorch
Setup complete.


## Run the experiments



In [36]:
import subprocess

try:

    for i, exp in enumerate(experiments, start=1):

        lr = exp["learning_rate"]
        d = exp["d"]

        print("\n" + "=" * 70)
        print(f"EXPERIMENT {i}/6")
        print(f"Learning rate: {lr}")
        print(f"Embedding dimension (d): {d}")
        print("=" * 70)

        # Start from the original config
        config = json.loads(json.dumps(original_config))

        config["train_config"]["learning_rate"] = lr
        config["sakt"]["d"] = d

        # Write configuration
        with open(CONFIG_PATH, "w") as f:
            json.dump(config, f, indent=4)

        # Run training
        process = subprocess.Popen(
            [
                "python",
                "-u",
                "train.py",
                "--model_name=sakt",
                "--dataset_name=ASSIST2009"
            ],
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)

        process.wait()

        print(f"\nReturn code: {process.returncode}")

        # If training exits abnormally, stop the entire experiment loop
        if process.returncode != 0:
            raise RuntimeError(
                f"Experiment {i} failed with exit code "
                f"{process.returncode}"
            )

        # Load AUC history
        auc_path = os.path.join(
            REPO_DIR,
            "ckpts",
            "sakt",
            "ASSIST2009",
            "aucs.pkl"
        )

        if not os.path.exists(auc_path):
            raise FileNotFoundError(
                f"aucs.pkl was not created for experiment {i}"
            )

        with open(auc_path, "rb") as f:
            aucs = pickle.load(f)

        # Calculate results
        best_auc = max(aucs)
        best_epoch = aucs.index(best_auc) + 1
        final_auc = aucs[-1]

        result = {
            "Experiment": i,
            "Learning Rate": lr,
            "d": d,
            "Best AUC": best_auc,
            "Best Epoch": best_epoch,
            "Final AUC": final_auc
        }

        results.append(result)

        # Display results immediately
        results_df = pd.DataFrame(results)

        print("\nExperiment completed.")
        print(f"Best AUC:   {best_auc:.6f}")
        print(f"Best Epoch: {best_epoch}")
        print(f"Final AUC:  {final_auc:.6f}")

        display(results_df)

        # Save after EVERY experiment
        results_df.to_csv(
            RESULTS_PATH,
            index=False
        )

        print(f"Results saved to: {RESULTS_PATH}")

except KeyboardInterrupt:
    print("\nTraining interrupted.")
    print("The experiment loop has stopped completely.")
    print("No additional configurations will be started.")

except Exception as e:
    print(f"\nExperiment stopped because of an error:")
    print(e)

finally:
    # Always restore the original config
    with open(CONFIG_PATH, "w") as f:
        json.dump(original_config, f, indent=4)

    print("\nOriginal config.json restored.")


EXPERIMENT 1/6
Learning rate: 0.001
Embedding dimension (d): 65
/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))
Epoch: 1,   AUC: 0.7087352308458417,   Loss Mean: 0.6262502670288086
Epoch: 2,   AUC: 0.7614535836057177,   Loss Mean: 0.5597079992294312
Epoch: 3,   AUC: 0.7780138348928661,   Loss Mean: 0.5352844595909119
Epoch: 4,   AUC: 0.7865180529618282,   Loss Mean: 0.5239111185073853
Epoch: 5,   AUC: 0.7907121506585186,   Loss Mean: 0.5175146460533142
Epoch: 6,   AUC: 0.7947051530542038,   Loss Mean: 0.512439489364624
Epoch: 7,   AUC: 0.796548261909928,   Loss Mean: 0.5086390376091003
Epoch: 8,   AUC: 0.7985596492695662,   Loss Mean: 0.5072037577629089
Epoch: 9,   AUC: 0.79978733

,Experiment,Learning Rate,d,Best AUC,Best Epoch,Final AUC
0,1,0.001,65,0.807554,34,0.801294


Results saved to: /content/drive/MyDrive/education-ml-research/sakt_hyperparameter_results.csv

EXPERIMENT 2/6
Learning rate: 0.001
Embedding dimension (d): 125
/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))
Epoch: 1,   AUC: 0.7583082307794542,   Loss Mean: 0.5980345606803894
Epoch: 2,   AUC: 0.7871008418929024,   Loss Mean: 0.5308004021644592
Epoch: 3,   AUC: 0.7962368891251088,   Loss Mean: 0.5138627886772156
Epoch: 4,   AUC: 0.8004135464026039,   Loss Mean: 0.5070104002952576
Epoch: 5,   AUC: 0.8021430579948475,   Loss Mean: 0.5014625191688538
Epoch: 6,   AUC: 0.8045860041286221,   Loss Mean: 0.49812889099121094
Epoch: 7,   AUC: 0.8058269559876052,   Loss Mean: 0.49483084678649

,Experiment,Learning Rate,d,Best AUC,Best Epoch,Final AUC
0,1,0.001,65,0.807554,34,0.801294
1,2,0.001,125,0.808607,19,0.787771


Results saved to: /content/drive/MyDrive/education-ml-research/sakt_hyperparameter_results.csv

EXPERIMENT 3/6
Learning rate: 0.0005
Embedding dimension (d): 65
/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))
Epoch: 1,   AUC: 0.6141551646333697,   Loss Mean: 0.6583130359649658
Epoch: 2,   AUC: 0.7054300851865529,   Loss Mean: 0.6061605215072632
Epoch: 3,   AUC: 0.7411127147899458,   Loss Mean: 0.572497546672821
Epoch: 4,   AUC: 0.7613522311753957,   Loss Mean: 0.5516953468322754
Epoch: 5,   AUC: 0.772875983629174,   Loss Mean: 0.5385340452194214
Epoch: 6,   AUC: 0.7805092336460366,   Loss Mean: 0.5302847623825073
Epoch: 7,   AUC: 0.7850490454262463,   Loss Mean: 0.5244801640510559


...

Results saved to: /content/drive/MyDrive/education-ml-research/sakt_hyperparameter_results.csv

EXPERIMENT 5/6
Learning rate: 0.0001
Embedding dimension (d): 65
/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))
Epoch: 1,   AUC: 0.552166911487499,   Loss Mean: 0.6642418503761292
Epoch: 2,   AUC: 0.5818145883246123,   Loss Mean: 0.6393423080444336
Epoch: 3,   AUC: 0.6112452277714318,   Loss Mean: 0.6281858086585999
Epoch: 4,   AUC: 0.6386428232789763,   Loss Mean: 0.6187676787376404
Epoch: 5,   AUC: 0.6630747845175108,   Loss Mean: 0.6095010042190552
Epoch: 6,   AUC: 0.6840783726537512,   Loss Mean: 0.6008458733558655
Epoch: 7,   AUC: 0.7011118963807098,   Loss Mean: 0.5915141105651855

,Experiment,Learning Rate,d,Best AUC,Best Epoch,Final AUC
0,1,0.0010,65,0.807554,34,0.801294
1,2,0.0010,125,0.808607,19,0.787771
2,3,0.0005,65,0.807989,63,0.807061
3,4,0.0005,125,0.809417,27,0.797855
4,5,0.0001,65,0.805287,100,0.805287


Results saved to: /content/drive/MyDrive/education-ml-research/sakt_hyperparameter_results.csv

EXPERIMENT 6/6
Learning rate: 0.0001
Embedding dimension (d): 125
/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))
Epoch: 1,   AUC: 0.54824959563916,   Loss Mean: 0.678837239742279
Epoch: 2,   AUC: 0.6234278518022655,   Loss Mean: 0.6332276463508606
Epoch: 3,   AUC: 0.6823338163595973,   Loss Mean: 0.610404908657074
Epoch: 4,   AUC: 0.7165793915583523,   Loss Mean: 0.5895418524742126
Epoch: 5,   AUC: 0.7366851071334171,   Loss Mean: 0.5725151896476746
Epoch: 6,   AUC: 0.7505282279798069,   Loss Mean: 0.5594024062156677
Epoch: 7,   AUC: 0.7603162792339515,   Loss Mean: 0.5496392250061035
E

,Experiment,Learning Rate,d,Best AUC,Best Epoch,Final AUC
0,1,0.0010,65,0.807554,34,0.801294
1,2,0.0010,125,0.808607,19,0.787771
2,3,0.0005,65,0.807989,63,0.807061
3,4,0.0005,125,0.809417,27,0.797855
4,5,0.0001,65,0.805287,100,0.805287
5,6,0.0001,125,0.809785,97,0.809723


Results saved to: /content/drive/MyDrive/education-ml-research/sakt_hyperparameter_results.csv

Original config.json restored.


## Hyperparameter Tuning Results

Six SAKT configurations were evaluated on the ASSISTments 2009 dataset by varying the learning rate and embedding dimension (d). The learning rates tested were 0.001, 0.0005, and 0.0001, while the embedding dimensions tested were 65 and 125. All other hyperparameters were kept constant, including a batch size of 128, 100 training epochs, sequence length of 100, Adam optimization, 5 attention heads, and 0.2 dropout.

The best-performing configuration was a learning rate of 0.0001 with an embedding dimension of 125, achieving a best AUC of 0.809785 at epoch 97. The final AUC for this configuration was 0.809723, indicating that performance remained highly stable through the end of training. The second-best configuration used a learning rate of 0.0005 with d = 125, achieving a best AUC of 0.809417 at epoch 27. The remaining configurations achieved best AUC values ranging from 0.805287 to 0.808607.

Based on these results, the configuration with a learning rate of 0.0001 and d = 125 was selected as the best-performing SAKT model for subsequent analysis. This configuration provided the highest observed AUC among all six experiments.

An important observation was that the final AUC was sometimes lower than the best AUC reached during training. For example, the learning rate of 0.001 with d = 125 achieved a best AUC of 0.808607 at epoch 19 but finished with an AUC of 0.787771. This demonstrates that continued training does not necessarily improve predictive performance, even when training loss may continue to decrease. Therefore, the highest AUC achieved during training and its corresponding epoch were used to identify the best-performing configuration.

## SAKT Performance Compared with Published Results

The best-performing SAKT configuration in this study achieved an AUC of 0.809785 on the ASSISTments 2009 dataset. This result is very close to the 0.8106 ± 0.0008 AUC reported for SAKT on ASSISTment2009 by the PyTorch implementation used as a reference for this project. The repository reports SAKT with n = 100, d = 100, 5 attention heads, and 0.2 dropout.

The difference between the current result and that PyTorch benchmark is only 0.000815 AUC, or 0.0815 percentage points. Thus, the model's performance is well within a 2-percentage-point difference from the independently reported PyTorch implementation. The original SAKT paper reports a higher AUC of approximately 0.848 on ASSIST2009, but replication studies and implementations show that SAKT performance can vary substantially depending on implementation, preprocessing, and train/test methodology.

Overall, the 0.809785 AUC obtained in this study provides strong evidence that the implementation is functioning comparably to an established PyTorch SAKT implementation, making it a reasonable model to use for the subsequent reliability analysis. The reference implementation is available in the knowledge-tracing-collection-pytorch GitHub repository.